---

#Curso: Data Mining - Semana 14
##Reglas de asociación


Docente: Soledad Espezúa (s.espezual@up.edu.pe)

---

In [ ]:
# instala libreria apyori
!pip install -q apyori

In [ ]:
# Importa librerias importantes
import math
import matplotlib.pyplot as plt
import pandas as pd
from apyori import apriori
import seaborn as sns

## <font color="green">Carga de los datos</font>

In [ ]:
# carga el Dataframe con los datos de transacciones
store_data = pd.read_csv('store_data.csv', header=None)
store_data.head()

Cada fila del dataset representa una transacción de compra realizada por un cliente. Las columnas indican los ítems adquiridos en dicha transacción; sin embargo, las columnas no corresponden a productos fijos, sino a posiciones dentro de la compra. Por ejemplo, una misma columna puede contener distintos productos en diferentes transacciones. Los valores NaN representan valores nulos e indican que, en esa posición, no se registró ningún ítem adicional comprado.

In [ ]:
# ¿Cuántas transacciones (filas) y posiciones (columnas) tenemos?
print("Número de transacciones:", store_data.shape[0])
print("Número máximo de productos en una compra:", store_data.shape[1])

## <font color="green">Preparación de datos</font>


La librería `apyori` **no trabaja con tablas (DataFrames)**. Necesita los datos como una
**lista de listas**:

- La lista grande contiene todas las compras.
- Cada lista pequeña (interna) es una compra con sus productos.

Por ejemplo:
```python
[
  ['shrimp', 'almonds', 'avocado'],   # compra 1
  ['burgers', 'meatballs', 'eggs'],   # compra 2
  ['chutney']                         # compra 3
]
```

Para lograrlo, recorremos cada fila y nos quedamos solo con los productos reales,
**descartando los NaN**. De paso hacemos dos limpiezas que este dataset necesita:

- **Quitamos espacios sobrantes.** El producto *asparagus* aparece una vez escrito como `' asparagus'`
  (con un espacio al inicio); sin limpiar, Apriori lo trataría como un producto distinto.
- **No repetimos un producto dentro de la misma compra.** En 5 compras aparece `ham` dos veces;
  en una transacción solo importa si el producto está o no está.

In [ ]:
# Convertimos la tabla a una lista de listas, ignorando los valores vacíos (NaN)
records = []

for fila in store_data.values:          # recorremos cada compra
    productos = []
    for item in fila:
        if pd.notna(item):              # solo si NO es un valor vacío (NaN)
            item = item.strip()         # quitamos espacios sobrantes: ' asparagus' -> 'asparagus'
            if item not in productos:   # evitamos repetir un producto en la misma compra
                productos.append(item)
    records.append(productos)           # agregamos la compra a la lista grande

print("Total de compras preparadas:", len(records))
print("Productos distintos:", len({p for compra in records for p in compra}))

In [ ]:
records[:5]

### <font color="green">Extracción de reglas de asociación con Apriori</font>

* **Soporte**: Qué tan frecuente es un producto o una combinación de productos.

  Ejm: Si 45 de 7500 compras incluyen pan → soporte = 0.006
* **Confianza**:  Si compran A, ¿qué tan probable es que compren B?.

   Ejm: "Si compran pan, el 70% también compra mantequilla"
 * **Lift**:  Cuánto cambia la probabilidad de B cuando aparece A

    Ejm: Lift = 3 → comprar A triplica la probabilidad de comprar B

### <font color=#e34a33>¿Cómo elegimos los parámetros (filtros) para este ejemplo?</font>
* `min_support` (soporte mínimo): descarta combinaciones de productos (*itemsets*) poco frecuentes
* `min_confidence` (confianza mínima): descarta reglas poco fiables
* `min_lift` (lift mínimo): nos quedamos solo con asociaciones fuertes
* `max_length`: Número máximo de productos por regla

Queremos reglas sobre combinaciones de productos que se vendan **unas 5 veces al día**.<br> Como el dataset
es de una semana (7 días), eso da unas **35 compras a la semana**.

1. **Soporte mínimo** = **35 / 7501 ≈ 0.0047** → cada *itemset* debe aparecer
   en al menos 35 compras. Como el soporte de una regla X ⇒ Y es el del *itemset* X ∪ Y, todas las reglas tendrán soporte ≥ 0.0047.
2. **Confianza** = **0.2** → de las compras que contienen el antecedente, al menos el **20%** debe contener
   también el consecuente.
3. **Lift** = **3** → solo asociaciones que al menos tripliquen la probabilidad.
4. **max_length** = **3** → reglas con hasta 3 productos.


> Cambiar estos valores y observar cómo cambian las reglas obtenidas
> (por ejemplo, un soporte mínimo de 0.0045 en la sección *Para practicar*).

In [ ]:
# Definimos los parámetros
compras_minimas   = 35                                 # ≈ 5 ventas al día × 7 días
soporte_minimo    = compras_minimas / len(records)     # 35 / 7501 ≈ 0.0047
confianza_minima  = 0.2
lift_minimo       = 3
max_productos     = 3

print(f"Soporte mínimo: {soporte_minimo:.4f} (aparecer en al menos {compras_minimas} compras)")

## <font color="green">Ejecutar el algoritmo Apriori</font>

`apyori` agrupa los resultados **por itemset**: cada elemento de `resultados` es una combinación de
productos que pasó los filtros, y dentro de ella (`ordered_statistics`) vienen **una o más reglas**
armadas con esos productos. Por eso el número de reglas puede ser mayor que el número de itemsets.

In [ ]:
# Ejecutamos Apriori con los parámetros elegidos
reglas = apriori(records,
                 min_support=soporte_minimo,
                 min_confidence=confianza_minima,
                 min_lift=lift_minimo,
                 max_length=max_productos)

# Apriori devuelve un generador; lo convertimos a lista para poder recorrerlo
resultados = list(reglas)

num_reglas = 0
for r in resultados:
    num_reglas = num_reglas + len(r.ordered_statistics)   # cuántas reglas salieron de este itemset

print("Itemsets que cumplen los filtros:", len(resultados))
print("Reglas encontradas:", num_reglas)

### <font color="blue">Cómo leer un resultado de apyori</font>

Antes de mostrarlas todas, veamos el primer resultado.

In [ ]:
resultados[0] # Difícil de interpretar

In [ ]:
# Miramos la estructura del primer resultado de forma más interpretable
primera = resultados[0]                # primer itemset
print("Productos del itemset:  ", sorted(primera.items))
print("Soporte:  ", round(primera.support, 4))
print("Número de reglas:       ", len(primera.ordered_statistics))

regla = primera.ordered_statistics[0]  # primera regla armada con ese itemset (índice 0)
print("Si compra ", sorted(regla.items_base),"→También compra", sorted(regla.items_add))

print("Confianza:", round(regla.confidence, 4))
print("Lift:     ", round(regla.lift, 2))

> **Importante:** <br>
> `items` es un conjunto (`frozenset`): no tiene orden, y el orden en que Python lo muestra puede cambiar de una sesión a otra.<br>
> La dirección de la regla (qué es antecedente y qué es consecuente) se lee de
> `items_base` y `items_add`<br>


### <font color=#c51b8a>Interpretación de esta regla: mushroom cream sauce → escalope</font>

- **Soporte = 0.0057** → el 0.57% de **todas** las compras contiene *mushroom cream sauce* **y** *escalope*
  juntos (43 de 7,501). Se calcula: nº de compras con ambos productos ÷ total de compras.

- **Confianza = 0.30** → de las compras que tienen *mushroom cream sauce*, el **30%** también incluye *escalope*.

- **Lift = 3.79** → quien compra *mushroom cream sauce* tiene **3.79 veces más probabilidad** de comprar
  *escalope* que un cliente cualquiera. Como el lift es mucho mayor que 1, **la asociación es fuerte**.

**Lectura del negocio:** la salsa de champiñones es un acompañamiento típico del escalope, así que es probable
que ambos productos se compren para la misma receta. El supermercado podría **colocarlos cerca** o crear una
promoción conjunta.

### <font color="blue">Verificamos la regla</font>

Calculamos las tres medidas con las fórmulas vistas en clase, contando compras directamente:

- soporte = σ(A ∪ B) / N
- confianza = σ(A ∪ B) / σ(A)
- lift = confianza / soporte(B)

In [ ]:
A, B = 'mushroom cream sauce', 'escalope'
N = len(records)

n_A  = sum(A in compra for compra in records)                    # compras con mushroom cream sauce
n_B  = sum(B in compra for compra in records)                    # compras con escalope
n_AB = sum(A in compra and B in compra for compra in records)    # compras con ambos

soporte   = n_AB / N
confianza = n_AB / n_A
lift      = confianza / (n_B / N)

print(f"Compras con {A}: {n_A} | con {B}: {n_B} | con ambos: {n_AB} | total: {N}")
print("Soporte:  ", round(soporte, 4))
print("Confianza:", round(confianza, 4))
print("Lift:     ", round(lift, 2))

### <font color="blue">Un itemset puede generar varias reglas</font>

El resultado en la posición 6 es el itemset {chocolate, frozen vegetables, shrimp}. Con esos tres productos
se armaron **dos reglas**: las dos tienen el **mismo soporte** (el del itemset), pero **distinta confianza**,
porque cambia el antecedente.

In [ ]:
otro = resultados[6]
print("Productos del itemset:", sorted(otro.items))
print("Soporte del itemset:  ", round(otro.support, 4))
print("Número de reglas:     ", len(otro.ordered_statistics))
print()

for regla in otro.ordered_statistics:            # recorremos ordered_statistics[0], [1], ...
    print(sorted(regla.items_base), "→", sorted(regla.items_add),
          "| confianza:", round(regla.confidence, 3), "| lift:", round(regla.lift, 2))

## <font color="green">Mostrar todas las reglas de forma ordenada</font>

1. Extraer los datos de **cada regla** a una tabla (DataFrame). Recorremos todas las reglas de cada
   itemset (no solo la primera) y tomamos la dirección de `items_base` → `items_add`.
2. Ordenarla por **lift** (de mayor a menor), para ver primero las asociaciones más fuertes.

In [ ]:
def tabla_de_reglas(resultados):
    filas = []
    for itemset in resultados:
        for regla in itemset.ordered_statistics:     # un itemset puede generar varias reglas
            if not regla.items_base:                  # omitimos reglas sin antecedente (∅ → X)
                continue
            si_compra      = ", ".join(sorted(regla.items_base))   # antecedente
            tambien_compra = ", ".join(sorted(regla.items_add))    # consecuente
            filas.append({
                "Regla":          f"{si_compra} → {tambien_compra}",
                "Si compra":      si_compra,
                "También compra": tambien_compra,
                "Soporte":        round(itemset.support, 4),
                "Confianza":      round(regla.confidence, 3),
                "Lift":           round(regla.lift, 2),
            })
    return pd.DataFrame(filas).sort_values("Lift", ascending=False).reset_index(drop=True)

tabla_reglas = tabla_de_reglas(resultados)
print("Número de reglas:", len(tabla_reglas))
tabla_reglas[["Regla", "Soporte", "Confianza", "Lift"]]

### <font color="blue">Otra forma de representar las reglas</font>

La misma tabla, separando el antecedente (*Si compra*) del consecuente (*También compra*).

In [ ]:
tabla_reglas[["Si compra", "También compra", "Soporte", "Confianza", "Lift"]]

### <font color="blue">Visualización de las reglas</font>

Cada punto es una regla: el eje horizontal es el soporte, el vertical la confianza y el color el lift.
Las reglas más interesantes suelen estar arriba (confianza alta) y con color intenso (lift alto).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.scatterplot(data=tabla_reglas, x="Soporte", y="Confianza",
                hue="Lift", palette="viridis", s=60,
                edgecolor="black", linewidth=0.4, legend=False, ax=ax)
ax.set_title("Reglas de asociación encontradas")
ax.grid(alpha=0.3)
plt.show()

## <font color="green">Para practicar</font>

Prueba a cambiar los parámetros. Algunas ideas:

1. Baja la **confianza** a `0.1`. ¿Aparecen más o menos reglas? ¿Por qué?
2. Sube el **lift mínimo** a `4`. ¿Cuáles reglas sobreviven?
3. Cambia `max_length` a `2`. ¿Qué tipo de reglas desaparecen?
4. Baja el **lift mínimo** a `1`. `apyori` genera también la regla ∅ → {mineral water}, con confianza
   0.238 y lift 1. ¿Por qué su confianza es igual al soporte de *mineral water*? ¿Por qué la función
   `tabla_de_reglas` la omite?
5. Cambia el **soporte mínimo** a `0.0045` (escribe `soporte_minimo = 0.0045`). ¿Cuántas compras equivale?
   ¿Qué regla nueva aparece? ¿Por qué no estaba con 0.0047?
6. Mira la tabla final: ¿qué dos productos tienen el **lift más alto**? ¿Qué decisión de
   negocio tomarías con esa información?

> **Recuerda:** un soporte alto = combinación frecuente; una confianza alta = regla fiable;
> un lift mayor que 1 = los productos se asocian más de lo esperado por azar.